# Contact Prediction
Supports CASP14, CASP15, and CAMEO datasets.

## Dependencies

In [ ]:
# ! pip install amplify scikit-learn biopython

In [ ]:
import os
import glob
import numpy as np

import torch
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import PPBuilder

from utils import load_pickle_dataset, load_from_hf, load_from_mila, apc, symmetrize

## Arguments

In [ ]:
# Model
source = "mila"
model_name = "AMPLIFY_350M"
model_path = "../checkpoints/AMPLIFY_350M/pytorch_model.pt"
tokenizer_path = None
config_path = "../checkpoints/AMPLIFY_350M/config.yaml"
device = "cuda"
compile = False
fp16 = True
batch_size = 256
method = "Jacobian"  # "Attention", "Jacobian"

# Dataset
# data_format options:
#   "pickle" — original CASP14 .pickle format
#   "casp"   — raw CASP15 directory: expects <data_path>/<target>/<target>.fasta + <target>.pdb
#   "cameo"  — raw CAMEO tar extract: expects <data_path>/<target>/target.fasta + target.pdb
data_format = "cameo"           # "pickle" | "casp" | "cameo"
data_name = "CAMEO_Jan_Mar_2024"
data_path = "../data/cameo_jan_mar_2024"   # path to extracted tar directory
n_proteins = None   # None = use all; set an int to cap (useful for quick tests)
max_length = 1024

# Logistic Regression (Attention method only)
n_train_samples = 20
threshold_c_beta = 8   # Cβ-Cβ distance threshold in Angstroms
min_sep = 6
l1_penalty = 0.15
seed = 0

# Log
output_file = f"../outputs/AMPLIFY_Contact_{data_name}.csv"

## Data Loading

### PDB parser (shared by CASP and CAMEO)

In [ ]:
def get_cb_coords(chain):
    """Return Cβ coordinates for each residue in a Bio.PDB chain.
    Glycine has no Cβ — use Cα as fallback.
    Returns an array of shape (L, 3) and the one-letter sequence string.
    """
    from Bio.PDB.Polypeptide import protein_letters_3to1
    coords, seq = [], []
    for res in chain.get_residues():
        if res.get_id()[0] != " ":   # skip HETATMs and water
            continue
        resname = res.get_resname().strip()
        try:
            letter = protein_letters_3to1[resname]
        except KeyError:
            continue
        if "CB" in res:
            coords.append(res["CB"].get_vector().get_array())
        elif "CA" in res:
            coords.append(res["CA"].get_vector().get_array())
        else:
            continue
        seq.append(letter)
    return np.array(coords, dtype=np.float32), "".join(seq)


def cb_distance_matrix(coords):
    """Compute pairwise Cβ distance matrix from (L, 3) coords array."""
    diff = coords[:, None, :] - coords[None, :, :]   # (L, L, 3)
    return np.sqrt((diff ** 2).sum(-1))               # (L, L)


def parse_fasta(path):
    """Parse the first sequence from a FASTA file."""
    seq = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq:  # only take the first record
                    break
            else:
                seq.append(line)
    return "".join(seq)


def load_pdb_target(pdb_path, max_length=None):
    """Parse a PDB file and return (sequence, distance_matrix).
    Uses the first chain. Truncates to max_length if provided.
    Returns (None, None) on parse failure.
    """
    parser = PDBParser(QUIET=True)
    try:
        structure = parser.get_structure("target", pdb_path)
    except Exception:
        return None, None

    chains = list(structure.get_chains())
    if not chains:
        return None, None

    coords, seq = get_cb_coords(chains[0])
    if len(seq) < 10:   # skip trivially short fragments
        return None, None

    if max_length and len(seq) > max_length:
        coords = coords[:max_length]
        seq = seq[:max_length]

    dist = cb_distance_matrix(coords)
    return seq, dist

### CAMEO loader

Expected directory layout after extracting the tar:
```
<data_path>/
  <target_id>/
    target.fasta
    target.pdb
```

In [ ]:
def load_cameo_dataset(data_path, max_length=None, n_proteins=None):
    """Load CAMEO targets from an extracted tar directory.

    Args:
        data_path: path to extracted CAMEO directory
        max_length: truncate sequences longer than this
        n_proteins: cap total number of targets (None = all)

    Returns:
        labels, proteins, dist_matrices
    """
    labels, proteins, dist_matrices = [], [], []

    target_dirs = sorted([
        d for d in glob.glob(os.path.join(data_path, "*"))
        if os.path.isdir(d)
    ])

    print(f"Found {len(target_dirs)} CAMEO target directories.")

    for target_dir in target_dirs:
        if n_proteins and len(labels) >= n_proteins:
            break

        target_id = os.path.basename(target_dir)
        pdb_path = os.path.join(target_dir, "target.pdb")
        fasta_path = os.path.join(target_dir, "target.fasta")

        if not os.path.exists(pdb_path):
            print(f"  Skipping {target_id}: no target.pdb found.")
            continue

        seq, dist = load_pdb_target(pdb_path, max_length=max_length)
        if seq is None:
            print(f"  Skipping {target_id}: PDB parse failed.")
            continue

        # If a fasta exists, prefer its sequence (handles non-standard residues better)
        if os.path.exists(fasta_path):
            fasta_seq = parse_fasta(fasta_path)
            if fasta_seq and len(fasta_seq) == len(seq):
                seq = fasta_seq

        labels.append(target_id)
        proteins.append(seq)
        dist_matrices.append(dist)

    print(f"Loaded {len(labels)} CAMEO targets.")
    return labels, proteins, dist_matrices

### CASP loader (CASP14 / CASP15)

Expected directory layout:
```
<data_path>/
  <TARGET_ID>/
    <TARGET_ID>.fasta   (or *.fasta)
    <TARGET_ID>.pdb     (or *.pdb)
```

In [ ]:
def load_casp_dataset(data_path, max_length=None, n_proteins=None):
    """Load CASP targets from a directory of per-target subdirectories.

    Args:
        data_path: path to CASP directory
        max_length: truncate sequences longer than this
        n_proteins: cap total number of targets (None = all)

    Returns:
        labels, proteins, dist_matrices
    """
    labels, proteins, dist_matrices = [], [], []

    target_dirs = sorted([
        d for d in glob.glob(os.path.join(data_path, "*"))
        if os.path.isdir(d)
    ])

    # Also support flat directory (all PDB files directly in data_path)
    if not target_dirs:
        pdb_files = glob.glob(os.path.join(data_path, "*.pdb"))
        target_dirs = []  # will fall through to flat-file handling below
        for pdb_path in sorted(pdb_files):
            if n_proteins and len(labels) >= n_proteins:
                break
            target_id = os.path.splitext(os.path.basename(pdb_path))[0]
            seq, dist = load_pdb_target(pdb_path, max_length=max_length)
            if seq is None:
                print(f"  Skipping {target_id}: PDB parse failed.")
                continue
            labels.append(target_id)
            proteins.append(seq)
            dist_matrices.append(dist)
        print(f"Loaded {len(labels)} CASP targets (flat layout).")
        return labels, proteins, dist_matrices

    print(f"Found {len(target_dirs)} CASP target directories.")

    for target_dir in target_dirs:
        if n_proteins and len(labels) >= n_proteins:
            break

        target_id = os.path.basename(target_dir)

        # Find the PDB file — try exact match first, then any .pdb
        pdb_path = os.path.join(target_dir, f"{target_id}.pdb")
        if not os.path.exists(pdb_path):
            candidates = glob.glob(os.path.join(target_dir, "*.pdb"))
            if not candidates:
                print(f"  Skipping {target_id}: no .pdb found.")
                continue
            pdb_path = candidates[0]

        seq, dist = load_pdb_target(pdb_path, max_length=max_length)
        if seq is None:
            print(f"  Skipping {target_id}: PDB parse failed.")
            continue

        # Optionally override sequence from FASTA
        fasta_path = os.path.join(target_dir, f"{target_id}.fasta")
        if not os.path.exists(fasta_path):
            candidates = glob.glob(os.path.join(target_dir, "*.fasta"))
            if candidates:
                fasta_path = candidates[0]
        if os.path.exists(fasta_path):
            fasta_seq = parse_fasta(fasta_path)
            if fasta_seq and len(fasta_seq) == len(seq):
                seq = fasta_seq

        labels.append(target_id)
        proteins.append(seq)
        dist_matrices.append(dist)

    print(f"Loaded {len(labels)} CASP targets.")
    return labels, proteins, dist_matrices

## Load Dataset

In [ ]:
if data_format == "pickle":
    # Original CASP14 pickle format
    labels, proteins, dist_matrices = load_pickle_dataset(data_path, n_proteins, max_length)
elif data_format == "cameo":
    labels, proteins, dist_matrices = load_cameo_dataset(data_path, max_length=max_length, n_proteins=n_proteins)
elif data_format == "casp":
    labels, proteins, dist_matrices = load_casp_dataset(data_path, max_length=max_length, n_proteins=n_proteins)
else:
    raise ValueError(f"Unknown data_format '{data_format}'. Choose from: pickle, cameo, casp.")

print(f"Dataset: {data_name} | {len(labels)} proteins")
lengths = [len(p) for p in proteins]
print(f"Sequence length — min: {min(lengths)}, max: {max(lengths)}, mean: {np.mean(lengths):.0f}")

## Train / Test Split

In [ ]:
# For Jacobian method no training split is needed — all proteins go to test
if method == "Attention":
    labels_train, labels_test, proteins_train, proteins_test, dist_matrices_train, dist_matrices_test = train_test_split(
        labels, proteins, dist_matrices, train_size=n_train_samples, random_state=seed
    )
else:
    labels_train, proteins_train, dist_matrices_train = [], [], []
    labels_test, proteins_test, dist_matrices_test = labels, proteins, dist_matrices

# Compute binary contact maps from distance matrices (Cβ-Cβ < threshold_c_beta)
contact_maps_train = [dist < threshold_c_beta for dist in dist_matrices_train]
contact_maps_test  = [dist < threshold_c_beta for dist in dist_matrices_test]

print(f"Train: {len(proteins_train)} | Test: {len(proteins_test)}")

## Load Model

In [ ]:
if source == "hf":
    model, tokenizer = load_from_hf(model_path, tokenizer_path, fp16=fp16)
elif source == "mila":
    model, tokenizer = load_from_mila(model_path, config_path)
else:
    raise ValueError(f"Only 'hf' and 'mila' sources are supported, not '{source}'.")
model.to(device)
torch.compile(model, disable=~compile)

## Contact Prediction Methods

In [ ]:
def get_attn_map(model, tokenizer, protein, device, fp16):
    with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.float16, enabled=fp16):
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

        x = torch.as_tensor(tokenizer.encode(protein)).to(torch.long)
        x = x.unsqueeze(0).to(device)
        attn_map = model(x, output_attentions=True)["attentions"]
        attn_map = torch.stack(attn_map).detach().cpu()
        attn_map = attn_map.reshape(-1, x.size(-1), x.size(-1))
        attn_map = attn_map[:, 1:-1, 1:-1]  # remove <bos> and <eos>
        attn_map = apc(symmetrize(attn_map))
        attn_map = attn_map.permute(1, 2, 0)  # (L, L, n_heads*n_layers)
        return attn_map

In [ ]:
def compute_jacobian(model, tokenizer, protein, device, fp16, batch_size=32):
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    amino_acids_ids = tokenizer.encode(amino_acids, add_special_tokens=False)

    with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.float16, enabled=fp16):
        input = torch.as_tensor(tokenizer.encode(protein)).to(torch.long)
        length = len(protein)

        mutated_inputs = []
        for n in range(len(protein)):
            x = torch.tile(input, [20, 1])
            x[:, n] = torch.as_tensor(amino_acids_ids)
            mutated_inputs.append(x)
        mutated_inputs = torch.cat(mutated_inputs, dim=0)

        ref_logits = model(input.unsqueeze(0).to(device))["logits"].squeeze()
        ref_logits = ref_logits[..., 1:-1, amino_acids_ids].cpu().numpy().astype(np.float64)

        mutated_logits = []
        for batch in torch.split(mutated_inputs, batch_size):
            mutated_logits.append(model(batch.to(device))["logits"][..., 1:-1, amino_acids_ids])
        mutated_logits = (
            torch.cat(mutated_logits, dim=0).reshape(length, 20, length, 20).cpu().numpy().astype(np.float64)
        )

    jac = mutated_logits - ref_logits
    jac = (jac + jac.transpose(2, 3, 0, 1)) / 2
    for i in range(4):
        jac -= jac.mean(i, keepdims=True)
    jac = np.sqrt(np.square(jac).sum((1, 3)))
    np.fill_diagonal(jac, 0)

    # APC
    a1 = jac.sum(0, keepdims=True)
    a2 = jac.sum(1, keepdims=True)
    jac = jac - (a1 * a2) / jac.sum()
    np.fill_diagonal(jac, 0)

    return torch.tensor(jac).unsqueeze(-1)

## Evaluation Utilities

In [ ]:
# Adapted from https://github.com/facebookresearch/esm/blob/main/examples/contact_prediction.ipynb
def compute_precisions(
    predictions: torch.Tensor,
    targets: torch.Tensor,
    src_lengths: torch.Tensor = None,
    minsep: int = 6,
    maxsep: int = None,
    override_length: int = None,
):
    if isinstance(predictions, np.ndarray):
        predictions = torch.from_numpy(predictions)
    if isinstance(targets, np.ndarray):
        targets = torch.from_numpy(targets)
    if predictions.dim() == 2:
        predictions = predictions.unsqueeze(0)
    if targets.dim() == 2:
        targets = targets.unsqueeze(0)
    override_length = (targets[0, 0] >= 0).sum()

    if predictions.size() != targets.size():
        raise ValueError(
            f"Size mismatch. predictions: {predictions.size()}, targets: {targets.size()}"
        )
    device = predictions.device
    batch_size, seqlen, _ = predictions.size()
    seqlen_range = torch.arange(seqlen, device=device)

    sep = seqlen_range.unsqueeze(0) - seqlen_range.unsqueeze(1)
    sep = sep.unsqueeze(0)
    valid_mask = sep >= minsep
    valid_mask = valid_mask & (targets >= 0)

    if maxsep is not None:
        valid_mask &= sep < maxsep

    if src_lengths is not None:
        valid = seqlen_range.unsqueeze(0) < src_lengths.unsqueeze(1)
        valid_mask &= valid.unsqueeze(1) & valid.unsqueeze(2)
    else:
        src_lengths = torch.full([batch_size], seqlen, device=device, dtype=torch.long)

    predictions = predictions.masked_fill(~valid_mask, float("-inf"))

    x_ind, y_ind = np.triu_indices(seqlen, minsep)
    predictions_upper = predictions[:, x_ind, y_ind]
    targets_upper = targets[:, x_ind, y_ind]

    topk = seqlen if override_length is None else max(seqlen, override_length)
    indices = predictions_upper.argsort(dim=-1, descending=True)[:, :topk]
    topk_targets = targets_upper[torch.arange(batch_size).unsqueeze(1), indices]
    if topk_targets.size(1) < topk:
        topk_targets = F.pad(topk_targets, [0, topk - topk_targets.size(1)])

    cumulative_dist = topk_targets.type_as(predictions).cumsum(-1)
    gather_lengths = src_lengths.unsqueeze(1)
    if override_length is not None:
        gather_lengths = override_length * torch.ones_like(gather_lengths, device=device)

    gather_indices = (
        torch.arange(0.1, 1.1, 0.1, device=device).unsqueeze(0) * gather_lengths
    ).type(torch.long) - 1

    binned_cumulative_dist = cumulative_dist.gather(1, gather_indices)
    binned_precisions = binned_cumulative_dist / (gather_indices + 1).type_as(binned_cumulative_dist)

    return {
        "AUC": binned_precisions.mean(-1),
        "P@L": binned_precisions[:, 9],
        "P@L2": binned_precisions[:, 4],
        "P@L5": binned_precisions[:, 1],
    }


def evaluate_prediction(predictions: torch.Tensor, targets: torch.Tensor):
    """Evaluate predictions across local/short/medium/long contact ranges.
    Long-range (minsep=24) P@L is the primary metric reported by AMPLIFY.
    """
    contact_ranges = [
        ("local",  3,  6),
        ("short",  6,  12),
        ("medium", 12, 24),
        ("long",   24, None),   # LR P@L — primary metric
    ]
    metrics = {}
    for name, minsep, maxsep in contact_ranges:
        rangemetrics = compute_precisions(predictions, targets, minsep=minsep, maxsep=maxsep)
        for key, val in rangemetrics.items():
            metrics[f"{name}_{key}"] = val.item()
    return metrics

In [ ]:
# Adapted from https://github.com/facebookresearch/esm/blob/main/examples/contact_prediction.ipynb
def plot_contacts_and_predictions(predictions, contacts, ax, cmap="Blues", ms=1, title_text=True):
    if isinstance(predictions, torch.Tensor):
        predictions = predictions.detach().cpu().numpy()
    if isinstance(contacts, torch.Tensor):
        contacts = contacts.detach().cpu().numpy()
    if ax is None:
        ax = plt.gca()

    seqlen = contacts.shape[0]
    relative_distance = np.add.outer(-np.arange(seqlen), np.arange(seqlen))
    bottom_mask = relative_distance < 0
    masked_image = np.ma.masked_where(bottom_mask, predictions)
    invalid_mask = np.abs(np.add.outer(np.arange(seqlen), -np.arange(seqlen))) < 6
    predictions = predictions.copy()
    predictions[invalid_mask] = float("-inf")

    topl_val = np.sort(predictions.reshape(-1))[-seqlen]
    pred_contacts = predictions >= topl_val
    true_positives  = contacts & pred_contacts & ~bottom_mask
    false_positives = ~contacts & pred_contacts & ~bottom_mask
    other_contacts  = contacts & ~pred_contacts & ~bottom_mask

    ax.imshow(masked_image, cmap=cmap)
    ax.plot(*np.where(other_contacts),  "o", c="grey", ms=ms)
    ax.plot(*np.where(false_positives), "o", c="r",    ms=ms)
    ax.plot(*np.where(true_positives),  "o", c="b",    ms=ms)
    if title_text is not None:
        ax.set_title(title_text)
    ax.axis("square")
    ax.set_xlim([0, seqlen])
    ax.set_ylim([0, seqlen])

## Train Logistic Regression (Attention method only)

In [ ]:
if method == "Attention":
    X_train, y_train = [], []
    for protein, contact_map in zip(proteins_train, contact_maps_train):
        pos = np.arange(contact_map.shape[0])
        attn_map = get_attn_map(model, tokenizer, protein, device, fp16)
        diag_idx = np.expand_dims(pos, axis=0) - np.expand_dims(pos, axis=1) >= min_sep
        X_train.extend(attn_map[diag_idx, :].reshape(-1, attn_map.shape[-1]).to(torch.float32))
        y_train.extend(contact_map[diag_idx].reshape(-1))
    X_train = np.asarray(X_train, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=np.float64)

    clf = LogisticRegression(solver="liblinear", penalty="l1", C=l1_penalty)
    clf.fit(X_train, y_train)
    print("Logistic regression trained.")

## Predict and Evaluate

In [ ]:
import pandas as pd

all_scores = []

for i, (label, protein, y_true) in enumerate(zip(labels_test, proteins_test, contact_maps_test)):
    if method == "Attention":
        x = get_attn_map(model, tokenizer, protein, device, fp16)
        y_pred = clf.predict_proba(x.reshape(-1, x.shape[-1]))[:, 1].reshape(y_true.shape)
    elif method == "Jacobian":
        y_pred = compute_jacobian(model, tokenizer, protein, device, fp16, batch_size=batch_size).reshape(y_true.shape)

    scores = evaluate_prediction(y_pred, y_true)
    scores["label"] = label
    scores["length"] = len(protein)
    all_scores.append(scores)

    # Visualize every 3 proteins
    if i % 3 == 0:
        fig, axes = plt.subplots(figsize=(18, 6), ncols=3)
    plot_contacts_and_predictions(
        y_pred,
        y_true,
        ax=axes[i % 3],
        title_text=f"{label} (L={len(protein)})\nLR P@L: {scores['long_P@L']:0.1%}",
    )
    if i % 3 == 2:
        plt.tight_layout()
        plt.show()
        plt.close()

plt.tight_layout()
plt.show()
plt.close()

## Summary

In [ ]:
df = pd.DataFrame(all_scores)

# Primary metric: Long-Range P@L (matches AMPLIFY Table 7)
lr_pal = df["long_P@L"]
print(f"\n=== {data_name} | {model_name} | {method} ===")
print(f"N proteins : {len(df)}")
print(f"LR P@L     : {lr_pal.mean()*100:.1f} ± {lr_pal.std()*100:.1f}")
print(f"LR P@L/2   : {df['long_P@L2'].mean()*100:.1f} ± {df['long_P@L2'].std()*100:.1f}")
print(f"LR P@L/5   : {df['long_P@L5'].mean()*100:.1f} ± {df['long_P@L5'].std()*100:.1f}")
print()
print("All ranges (mean P@L):")
for rng in ["local", "short", "medium", "long"]:
    print(f"  {rng:8s}: {df[f'{rng}_P@L'].mean()*100:.1f}")

os.makedirs(os.path.dirname(output_file), exist_ok=True)
df.to_csv(output_file, index=False)
print(f"\nResults saved to {output_file}")